_HAMRIT Wassim_ 
_5A FISE D1_

# **Rapport - Utilisation de plusieurs méthodes pour compter l'occurence des mots d'un texte**


Dans ce rapport, on va étudier plusieurs méthodes permettant de compter les mots dans un fichier texte. <br><br>
On va s'intéresser à **3 méthodes** différentes: 

- Une méthode simple en utilisant du code Python en traitant les données de manière **séquetielle**.
- Une méthode **pseudo-MapReduce** qui reproduit les étapes du MapReduce (map, shuffle, reduce) en utilisant Python.
- Une méthode qui s'appuie sur le parallélisme en utilisant **Hadoop MapReduce**.

Pour chacune de ces méthodes, on va utiliser plusieurs fichiers texte en faisant varier leur nombre de lignes:
- **N = 100 lignes** 
- **N = 100 000 lignes** 
- **N = 5 000 000 lignes** 
- **N = 50 000 000 lignes** 



En utilisant ces fichiers texte de différentes tailles, on pourra donc étudier les différences de performance entre chaque méthodes selon la quantité des données.

## **1 - Méthode séquentielle avec Python**


#### a) Fichier de taille N = 100 lignes

In [24]:
import time

def compter_occurrences(fichier):
    with open (fichier , 'r', encoding='utf -8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot , 0) + 1
    return compteur

debut = time.perf_counter();

fichier = "./textes/fichier1_100.txt"
resultats = compter_occurrences(fichier)
for mot , nb in sorted (resultats.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

chien : 25
chat : 24
maison : 20
voiture : 16
arbre : 15

Temps d'exécution : 0.00155s


#### b) Fichier de taille N = 100 000 lignes

In [25]:
import time

def compter_occurrences(fichier):
    with open (fichier , 'r', encoding='utf -8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot , 0) + 1
    return compteur

debut = time.perf_counter();

fichier = "./textes/fichier2_100k.txt"
resultats = compter_occurrences(fichier)
for mot , nb in sorted (resultats.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

voiture : 20101
chien : 20036
arbre : 20035
chat : 19922
maison : 19906

Temps d'exécution : 0.02629s


#### c) Fichier de taille N = 5 000 000 lignes

In [26]:
import time

def compter_occurrences(fichier):
    with open (fichier , 'r', encoding='utf -8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot , 0) + 1
    return compteur

debut = time.perf_counter();

fichier = "./textes/fichier3_5M.txt"
resultats = compter_occurrences(fichier)
for mot , nb in sorted (resultats.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

chien : 1000508
chat : 1000123
arbre : 1000102
maison : 999890
voiture : 999377

Temps d'exécution : 0.77141s


#### d) Fichier de taille N = 50 000 000 lignes

In [27]:
import time

def compter_occurrences(fichier):
    with open (fichier , 'r', encoding='utf -8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot , 0) + 1
    return compteur

debut = time.perf_counter();

fichier = "./textes/fichier4_50M.txt"
resultats = compter_occurrences(fichier)
for mot , nb in sorted (resultats.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

chat : 10003244
voiture : 10002309
chien : 9999121
maison : 9998452
arbre : 9996874

Temps d'exécution : 8.36464s


#### Comparaison du temps d'exécution selon le nombre de lignes (séquentiel) :

| Fichier | Nombre de lignes | Temps d'exécution (secondes) |
| :--- | :---: | :---: |
| **fichier1_100.txt** | 100 | 0.00155s |
| **fichier2_100k.txt** | 100 000 | 0.02629s |
| **fichier3_5M.txt** | 5 000 000 | 0.77141s |
| **fichier4_50M.txt** | 50 000 000 | 8.36464s |

## **2 - Méthode pseudo-MapReduce avec Python**


#### a) Fichier de taille N = 100 lignes

In [30]:
import time
from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

debut = time.perf_counter();

fichier = "./textes/fichier1_100.txt"
mapped = []
    
with open(fichier, "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))

grouped = shuffle(mapped)
reduced = reducer(grouped)

for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {count}")


fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

chien : 25
chat : 24
maison : 20
voiture : 16
arbre : 15

Temps d'exécution : 0.00575s


#### b) Fichier de taille N = 100 000 lignes

In [31]:
import time
from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

debut = time.perf_counter();

fichier = "./textes/fichier2_100k.txt"
mapped = []
    
with open(fichier, "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))

grouped = shuffle(mapped)
reduced = reducer(grouped)

for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {count}")


fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

voiture : 20101
chien : 20036
arbre : 20035
chat : 19922
maison : 19906

Temps d'exécution : 0.04110s


#### c) Fichier de taille N = 5 000 000 lignes

In [32]:
import time
from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

debut = time.perf_counter();

fichier = "./textes/fichier3_5M.txt"
mapped = []
    
with open(fichier, "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))

grouped = shuffle(mapped)
reduced = reducer(grouped)

for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {count}")


fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

chien : 1000508
chat : 1000123
arbre : 1000102
maison : 999890
voiture : 999377

Temps d'exécution : 2.09603s


#### d) Fichier de taille N = 50 000 000 lignes

In [33]:
import time
from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

debut = time.perf_counter();

fichier = "./textes/fichier4_50M.txt"
mapped = []
    
with open(fichier, "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))

grouped = shuffle(mapped)
reduced = reducer(grouped)

for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {count}")


fin = time.perf_counter();
    
temps_exec = fin - debut

print(f"\nTemps d'exécution : {temps_exec:.5f}s")

chat : 10003244
voiture : 10002309
chien : 9999121
maison : 9998452
arbre : 9996874

Temps d'exécution : 28.78739s


#### Comparaison du temps d'exécution selon le nombre de lignes (pseudo-MapReduce) :

| Fichier | Nombre de lignes | Temps d'exécution (secondes) |
| :--- | :---: | :---: |
| **fichier1_100.txt** | 100 | 0.00575s |
| **fichier2_100k.txt** | 100 000 | 0.04110s |
| **fichier3_5M.txt** | 5 000 000 | 2.09603s |
| **fichier4_50M.txt** | 50 000 000 | 28.78739s |

## **3 - Méthode Hadoop MapReduce**


##### Installation de Java & Hadoop


```bash 
wget https://downloads.apache.org/hadoop/common/hadoop-3.4.0/hadoop-3.4.0.tar.gz
tar -xvzf hadoop-3.4.0.tar.gz
mv hadoop-3.4.0 ~/hadoop

# Configuration de l'environnement
echo 'export HADOOP_HOME=$HOME/hadoop' >> ~/.bashrc
source ~/.bashrc

# Démarrage du cluster
hdfs namenode -format
start-dfs.sh
start-yarn.sh

hdfs dfs -mkdir -p /user/oxfrz/input
hdfs dfs -put fichier1_100.txt /user/oxfrz/input/
hdfs dfs -put fichier2_100k.txt /user/oxfrz/input/
hdfs dfs -put fichier3_5M.txt /user/oxfrz/input/
hdfs dfs -put fichier4_50M /user/oxfrz/input/
```